# Key Biodiversity Areas (KBA)

This notebook loads the Key Biodiversity Areas (KBA) as gdf and assigns the score 1 to all poylgons. Than it rasterizes the polygons using the bii layer as reference grid, overlays it on country boundaries and creates:
- a raster (with bii as reference raster)
- a map figure (`OUT_PNG`)
  
## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `kba.gpkg`
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)


In [ ]:
# Configuration (edit these paths / settings)
KBA_GPKG = 'kba.gpkg'
BII_5000M_TIF = 'sensitivity/biodiversity_intactness/bii_5000m.tif'
KBA_RASTERIZED_TIF = 'kba_rasterized.tif'
WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG = 'World_Countries_(Generalized)_8414823838130214587.gpkg'
KBA_5000M_TIF = 'kba_5000m.tif'

In [ ]:
#import packages
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from rasterio.windows import Window
from rasterio.features import rasterize

In [ ]:
#load kba as gdf
gdf = gpd.read_file(KBA_GPKG)

In [ ]:
#add score column with value 1 indicating higher sensitivity
gdf["score"] = 1

In [ ]:
# rasterize kba variable using bii as reference raster

#paths
ref = BII_5000M_TIF #reference raster
out_path = KBA_RASTERIZED_TIF

# Open reference raster as the template grid
with rasterio.open(ref) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    out_shape = (src.height, src.width)

    # reproject polygons to match raster CRS
    gdf_r = gdf.to_crs(crs)
    
    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
        for block_index, window in src.block_windows(1):    
            win_transform = rasterio.windows.transform(window, transform)
        
            window_raster = rasterize(
                    [(geom, value) for geom, value in zip(gdf_r.geometry, gdf_r.score)],
                    out_shape=(window.height, window.width),  
                    transform=win_transform,
                    fill=-9999,
                    dtype="float32"
                )
    
            dst.write(window_raster, 1, window=window)

print("Saved:", out_path)

In [ ]:
#add country boundaries

#paths
kba_path = KBA_RASTERIZED_TIF  # your existing raster
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_path = KBA_5000M_TIF


# Open kba raster as the template grid
with rasterio.open(kba_path) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    src_nodata = src.nodata 

    # Read countries and project to raster CRS
    world = gpd.read_file(countries_path).to_crs(crs)

    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
        for block_index, window in src.block_windows(1):
            kba = src.read(1, window=window).astype("float32")
            win_transform = rasterio.windows.transform(window, transform)
            
            country_mask = rasterize(
                [(geom, 1) for geom in world.geometry],
                out_shape=(window.height, window.width),
                transform=win_transform,
                fill=0,
                dtype="uint8"
            )

            # Create output for this window
            out = np.full((window.height, window.width), -9999, dtype="float32")
            inside = country_mask == 1

            # inside countries: treat -9999 (no data) in kba as 0 (no kba)
            kba_inside = kba.copy()
            kba_inside[(kba_inside == -9999)] = 0.0
            out[inside] = kba_inside[inside]

            dst.write(out, 1, window=window)

out[inside] = kba_inside[inside]

print("Saved:", out_path)


In [ ]:
#plot

# paths
raster_path = KBA_5000M_TIF
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_png = "kba.png"


# load raster
with rasterio.open(raster_path) as src:
    arr = src.read(1)
    bounds = src.bounds
    crs = src.crs

# Mask -9999 (nodata) + treat 0 as transparent 
masked = np.ma.masked_where(arr != 1, arr)
    
cmap = ListedColormap(["#53C256"])  # will be used for the unmasked cells

# load countries and drop Antarctica 
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"].copy()

#build figure, set size and background color
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Countries 
world.plot(
    ax=ax,
    facecolor="#F6F7F9",   
    edgecolor="#B9C0C8",   
    linewidth=0.35,
    zorder=1
)

# raster overlay
ax.imshow(
    masked,
    cmap=cmap,
    vmin=1, vmax=1,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=0.95,
    zorder=2
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)

#set title
ax.set_title("Key Biodiversity Areas", fontsize=18, fontweight="semibold", pad=14)

#no axis
ax.set_axis_off()

#  legend
legend_handles = [Patch(facecolor="#53C256", edgecolor="none", label="KBA")]
leg = ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    framealpha=1,
    facecolor="white",
    edgecolor="#E3E6EA",
    borderpad=0.8,
    handlelength=1.2,
)

plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
